In [3]:
#get all the tables:
from db_setup import conn, currencyexchange, customer, date, product, sales, store
#instead of saving csv files in sql and then importing one by one
import pandas as pd


0- Cleanup our dates

In [4]:
#checked with df.info() and the date columns are strings. do converting all of them (even those i havent checked):

sales["orderdate"] = pd.to_datetime(sales["orderdate"])
sales["deliverydate"] = pd.to_datetime(sales["deliverydate"])
customer["birthday"] = pd.to_datetime(customer["birthday"])
customer["startdt"] = pd.to_datetime(customer["startdt"])
customer["enddt"] = pd.to_datetime(customer["enddt"])
store["opendate"] = pd.to_datetime(store["opendate"])
store["closedate"] = pd.to_datetime(store["closedate"])
date["date"] = pd.to_datetime(date["date"])
currencyexchange["date"] = pd.to_datetime(currencyexchange["date"])



In [5]:
sales2=sales.copy()

In [6]:
sales2["revenue_line_usd"]=sales2["quantity"]*sales2["netprice"]/sales2["exchangerate"]
sales2["cost_line_usd"]=sales2["quantity"]*sales2["unitcost"]/sales2["exchangerate"]
sales2["profit_line_usd"]=sales2["revenue_line_usd"]-sales2["cost_line_usd"]
sales2["delivery_days_line"]=(sales2["deliverydate"]-sales2["orderdate"]).dt.days

1-Order Base View

In [9]:
order_base=sales2.groupby("orderkey").agg(
    orderdate=("orderdate","min"),
    delivery_date=("deliverydate","max"),
    customerkey=("customerkey", "max"),
    storekey=("storekey", "max"),
    line_count=("linenumber", "count"),
    distinct_products=("productkey", "nunique"),
    total_units=("quantity", "sum"),
    order_revenue=("revenue_line_usd", "sum"),
    order_cost=("cost_line_usd", "sum"),
    order_profit=("profit_line_usd", "sum"),
    delivery_days=("delivery_days_line", "max")
    ).reset_index()


In [20]:
#saving the views as parquet files to be later used on other notebooks:

order_base.to_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\order_base.parquet", index=False)

2 - Order Enriched View:

In [10]:
customer2=customer.copy()
store2=store.copy()

Let's clean the names first. There are no missing values either for givenname and surname columns of the customer table. But I'll use fillna either way.

In [11]:
customer.isna().sum() 
#to get the number of NAs in each column
customer.isna().mean()*100
#to get the % of NAs in each column

#the same was done with store table

customerkey      0.00000
geoareakey       0.00000
startdt          0.00000
enddt            0.00000
continent        0.00000
gender           0.00000
title            0.00000
givenname        0.00000
middleinitial    0.00000
surname          0.00000
streetaddress    0.00000
city             0.00000
state            0.00000
statefull        0.00000
zipcode          0.00000
country          0.00000
countryfull      0.00000
birthday         0.00000
age              0.00000
occupation       0.00000
company          0.10001
vehicle          0.00000
latitude         0.00000
longitude        0.00000
dtype: float64

In [ ]:
customer2["customer_name"]=(customer2["givenname"].fillna("").str.strip() + " " + customer2["surname"].fillna("").str.strip()).str.strip()



In [13]:
order_enriched = (
    order_base
    .merge(
        customer2[[
            "customerkey", 
            "countryfull", 
            "gender", 
            "age",
            "occupation", 
            "birthday", 
            "customer_name"
        ]],
        on="customerkey",
        how="inner"
    )
    .merge(
        store2[[
            "storekey", 
            "countryname", 
            "state", 
            "status", 
            "squaremeters"
        ]],
        on="storekey",
        how="inner"
    )
)


add the age:
note that we have to consider the month and day of the birthday compared to the order date. 

In [14]:
order_enriched["age_at_order"] = (
    order_enriched["orderdate"].dt.year - order_enriched["birthday"].dt.year
    - (
        (order_enriched["orderdate"].dt.month < order_enriched["birthday"].dt.month) |
        (
            (order_enriched["orderdate"].dt.month == order_enriched["birthday"].dt.month) &
            (order_enriched["orderdate"].dt.day < order_enriched["birthday"].dt.day)
        )
    ).astype(int)
)

rename columns like the sql code:

In [15]:
order_enriched = order_enriched.rename(columns={
    "countryfull": "customer_country",
    "countryname": "store_country",
    "state": "store_state",
    "status": "store_status"
})

actual view with just the columns i want:

In [16]:
order_enriched_view = order_enriched[[
    "orderkey",
    "orderdate",
    "delivery_date",
    "customerkey",
    "storekey",
    "line_count",
    "distinct_products",
    "total_units",
    "order_revenue",
    "order_cost",
    "order_profit",
    "delivery_days",
    "customer_country",
    "gender",
    "age",
    "age_at_order",
    "occupation",
    "customer_name",
    "store_country",
    "store_state",
    "store_status",
    "squaremeters"
]]

order_enriched_view

,orderkey,orderdate,delivery_date,customerkey,storekey,line_count,distinct_products,total_units,order_revenue,order_cost,...,customer_country,gender,age,age_at_order,occupation,customer_name,store_country,store_state,store_status,squaremeters
0,1000,2015-01-01,2015-01-01,947009,400,2,2,2,1182.677889,685.196010,...,United Kingdom,male,27,20,Computer hardware engineer,Hayden Rowe,United Kingdom,Dungannon and South Tyrone,None,1300.0
1,1001,2015-01-01,2015-01-01,1772036,430,1,1,2,108.752000,50.008000,...,United States,female,50,44,Letterpress setter,Beverly Tejeda,United States,Alaska,None,1190.0
2,1002,2015-01-01,2015-01-01,1518349,660,4,4,15,3458.642260,1738.882000,...,United States,female,80,74,Leasing consultant,Kristi Daniel,United States,West Virginia,None,1785.0
3,1003,2015-01-01,2015-01-01,1317097,510,1,1,3,224.977500,103.455000,...,United States,male,29,23,Range scientist,Richard Holland,United States,Maine,None,1295.0
4,1004,2015-01-01,2015-01-01,254117,80,4,4,5,2419.632568,904.326130,...,Canada,male,76,69,Cinematographer,Jack Gabor,Canada,Newfoundland and Labrador,None,2105.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83125,3398031,2024-04-20,2024-04-20,1692049,530,4,4,13,2624.342200,1301.730000,...,United States,male,72,75,Skills training coordinator,George Wilson,United States,Montana,None,1260.0
83126,3398032,2024-04-20,2024-04-25,852158,999999,2,2,3,312.848194,165.217854,...,Netherlands,female,72,75,Organic chemist,Guus Doodeman,Online,Online,None,NaN
83127,3398033,2024-04-20,2024-04-20,635184,160,3,3,13,5235.110046,1959.241504,...,France,female,66,69,Director,Virginie Patel,France,Limousin,None,385.0
83128,3398034,2024-04-20,2024-04-21,664396,999999,3,3,9,1420.644615,739.181847,...,France,female,39,42,Media specialist,Karlotta Rivière,Online,Online,None,NaN


In [19]:
#saving the views as parquet files to be later used on other notebooks:

order_enriched.to_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\order_enriched.parquet", index=False)